# Style Beyond Authorship

**Computational Text Analysis Pack · Module 09 — Stylometry**

*Technique:* Stylometric features for genre, register and translation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/07-stylometry/09c_style_beyond_authorship.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv`, `african_legal.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv', 'african_legal.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
df_legal = pd.read_csv('african_legal.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Delta was built for authorship, but the underlying idea — that unconscious
frequency habits carry information — applies to anything that shapes writing:
genre, register, period, whether a text is a translation.

For African DH this last group matters most. Authorship attribution needs many
works by each candidate, which most African archives cannot supply. **Register and
genre analysis needs only two groups of documents**, and every archive has those.

## The tooling

**numpy** for the z-scored feature matrices and **scipy** for clustering.
Stylometry's field standard is **stylo** (R), which implements every Delta
variant plus bootstrap consensus trees; **faststylometry** is the Python
equivalent. We write Delta out because it is ten lines.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
# Three registers, same language, from three corpora.
def sample_docs(series, n, min_words=300, seed=0):
    rng = np.random.default_rng(seed)
    ok = [str(t) for t in series if len(str(t).split()) >= min_words]
    idx = rng.choice(len(ok), size=min(n, len(ok)), replace=False)
    return [ok[i] for i in idx]

groups = {
    "news":     sample_docs(df[df["language"] == "English"]["text"], 60),
    "colonial": sample_docs(df_colonial["text"], 60),
    "legal":    sample_docs(df_legal["text"], 60),
}
for k, v in groups.items():
    print(f"{k:<10}{len(v):>4} documents")

In [ ]:
docs, labels = [], []
for name, texts in groups.items():
    docs.extend(texts)
    labels.extend([name] * len(texts))
labels = np.array(labels)

MFW = 120
corpus_counts = Counter()
tokenised = []
for t in docs:
    toks = tokenise(t)
    tokenised.append(toks)
    corpus_counts.update(toks)
features = [w for w, _ in corpus_counts.most_common(MFW)]
rates = np.array([[toks.count(w) / len(toks) * 100 for w in features]
                  for toks in tokenised])
Z = (rates - rates.mean(0)) / (rates.std(0) + 1e-12)
print(f"{Z.shape[0]} documents × {Z.shape[1]} function words")

In [ ]:
from sklearn.decomposition import PCA

p = PCA(n_components=2).fit(Z)
coords = p.transform(Z)
fig, ax = plt.subplots(figsize=(8, 6))
for name in groups:
    m = labels == name
    ax.scatter(coords[m, 0], coords[m, 1], s=18, alpha=0.7, label=name)
ax.set_xlabel(f"PC1 ({p.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({p.explained_variance_ratio_[1]:.1%})")
ax.set_title("Three registers, separated by function words alone")
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# What the components are made of — the interpretable part of PCA.
for c in range(2):
    load = p.components_[c]
    hi = [features[i] for i in np.argsort(load)[::-1][:8]]
    lo = [features[i] for i in np.argsort(load)[:8]]
    print(f"PC{c + 1}  positive: {', '.join(hi)}")
    print(f"     negative: {', '.join(lo)}")
    print()
print("Read these as a stylistic axis. A component loading on 'the, of, in'")
print("against 'i, you, we' is a nominal/personal axis — formality, essentially.")

In [ ]:
# Non-lexical features: they cross languages, which word rates do not.
def style_features(text):
    toks = tokenise(text)
    sents = [s for s in re.split(r"[.!?]+", str(text)) if s.strip()]
    return {
        "mean_word_len": float(np.mean([len(w) for w in toks])) if toks else 0,
        "mean_sent_len": float(np.mean([len(s.split()) for s in sents])) if sents else 0,
        "ttr_200": len(set(toks[:200])) / 200 if len(toks) >= 200 else np.nan,
        "comma_rate": str(text).count(",") / max(len(toks), 1) * 100,
        "digit_rate": sum(c.isdigit() for c in str(text)) / max(len(str(text)), 1) * 100,
    }

feat = pd.DataFrame([style_features(t) for t in docs])
feat["register"] = labels
print(feat.groupby("register").mean(numeric_only=True)
          .to_string(float_format=lambda v: f"{v:.2f}"))

### 🔧 Your turn 1

Build the same non-lexical table for the four Nigerian languages in `df`.

These features need no vocabulary, so they work on any language. Which of them
differ across languages, and is that about the languages or about the outlets
that were sampled? (You met this confound in 02d — it has not gone away.)

## Where this becomes a research programme

Two projects this corpus supports directly, both under-served in the literature:

- **Register variation in African-language journalism.** Do the four Nigerian
  languages mark the news/opinion distinction the same way? Function-word
  profiles per language, compared across categories.
- **Translationese in the colonial archive.** Texts translated into English carry
  measurable traces of the source language. The colonial corpus contains both
  translated and originally-English material.

Neither needs a tool that does not exist for African languages. Both need a
carefully built corpus, which is the skill this course is actually teaching.

### 🔧 Your turn 2

Pick one of those two, or a version for your own material, and write the first
paragraph of the methods section: corpus, features, comparison, and the confound
you cannot remove.

That paragraph is the deliverable of the capstone (Module 12).

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Mean word length differs sharply across the four languages, and
it is genuinely linguistic — agglutinative morphology produces longer written
words. Sentence length and comma rate are far more about house style and the
outlet's conventions than about the language. The general rule: features
depending on the writing system and morphology are linguistic; features depending
on punctuation and layout are editorial. Keep them apart in any comparison.

**Your turn 2.** A strong opening paragraph names the corpus and its size, the
feature set with its parameter (120 MFW, or the five non-lexical measures), the
groups being compared, and one sentence conceding what else differs between those
groups. If you cannot write the last sentence, you have not yet found the
confound — and it is there.

</details>

## Where this stops

You can use stylometric features for questions beyond authorship. Next: **Module
10 — visualising text**, where the main lesson is which charts to refuse to make.